# Fine-Tuning TinyLlama with LoRA — Experiment Notebook

---

##What I'm experimenting with?

In this notebook, I'm experimenting with LoRA fine-tuning using TinyLlama.

The goal is to take a pre-trained language model and fine-tune it on customer-support examples to see how much its responses can be adapted toward a particular style and task.

I'm mainly using this as a hands-on experiment to understand:

how a pre-trained model is fine-tuned
what LoRA changes compared with full fine-tuning
how the training data is prepared
what happens to the model's responses after training
how useful the fine-tuned model is for a specific task

This is an experiment/learning notebook, so I'll be keeping track of the steps, outputs, and observations as I go.


## CELL 1 — Installing the needed tools

The main libraries I'm using here are:

* transformers — for working with pre-trained Hugging Face models
* peft — provides Parameter-Efficient Fine-Tuning methods such as LoRA
* trl — provides tools that simplify language-model fine-tuning
* datasets — for loading and preparing the training dataset
* accelerate — helps with running training efficiently on the available hardware
* bitsandbytes — provides memory-efficient model loading and quantization

In [ ]:
# -q means quiet mode — installs without showing every single detail

!pip install -q transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.3 MB/s eta 0:00:00


## CELL 2 — Import the needed tools


Here is what each import does:
- **AutoModelForCausalLM** — loads a causal language model such as TinyLlama for text generation (like GPT style)
- **AutoTokenizer** — Converts text into numbers that model can understands
- **BitsAndBytesConfig** — Settings for compressing the model to 4-bit so it fits in free Colab memory
- **LoraConfig** — Where we set our LoRA settings (rank, alpha etc.)
- **get_peft_model** — Takes our base model and wraps LoRA around it
- **SFTTrainer** — The trainer that handles the fine-tuning loop for us automatically
- **TrainingArguments** — Settings for training (how many steps, learning rate etc.)
- **load_dataset** — Loads our training data
- **torch** — PyTorch. The engine that runs all the maths underneath

In [ ]:
# Each 'from X import Y' means: from library X, bring in tool Y

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
import torch

# Let's confirm our GPU is available

print("Device available:", "cuda" if torch.cuda.is_available() else "cpu")

Device available: cuda


## CELL 3 — Creating training data

For this experiment, I need a small dataset of examples that shows the model how I want it to respond.

Each pair has:
- An **instruction** (the user's message)
- A **response** (what we want the model to say)

I'm formatting these examples using a chat-style format so that the training data has a consistent structure.

For this first experiment, I'm creating a small set of customer-support examples manually rather than using a large real-world dataset.

The main thing I'm exploring here is how the quality and structure of the examples affect the model's behaviour after fine-tuning.

In [ ]:
# These are our training examples
# Each example teaches the model one customer support interaction
# The more examples you have, the better the model learns

training_examples = [
    {
        "instruction": "My order has not arrived yet. It has been 10 days.",
        "response": "I apologize for the delay. Could you please share your order ID so I can check the status for you? Standard delivery takes 5-7 business days, but I will investigate this right away."
    },
    {
        "instruction": "I want to return a product I bought last week.",
        "response": "Of course! Our return window is 30 days from purchase. Please visit your Orders section, select the item, and click 'Return'. You will receive a prepaid label via email within 24 hours."
    },
    {
        "instruction": "How do I reset my account password?",
        "response": "To reset your password, click 'Forgot Password' on the login page and enter your registered email. You will receive a reset link within a few minutes. Please also check your spam folder if you don't see it."
    },
    {
        "instruction": "I was charged twice for the same order.",
        "response": "I sincerely apologize for this inconvenience. This should not have happened. Please share your order ID and the last 4 digits of the payment method used, and I will initiate a refund for the duplicate charge immediately."
    },
    {
        "instruction": "Can I change my delivery address after placing an order?",
        "response": "Address changes are possible within 1 hour of placing the order. If it has been less than an hour, please contact us immediately with your order ID. After 1 hour, the order goes to our warehouse and cannot be redirected."
    },
    {
        "instruction": "My product arrived damaged.",
        "response": "I am truly sorry to hear that. Please take a photo of the damaged item and packaging, then email it to support@store.com with your order ID. We will arrange a replacement or full refund within 2 business days."
    },
    {
        "instruction": "What are your customer support working hours?",
        "response": "Our customer support team is available Monday to Friday, 9 AM to 6 PM IST. For urgent issues outside these hours, you can email us and we will respond within 24 hours on the next business day."
    },
    {
        "instruction": "I never received my refund.",
        "response": "Refunds typically take 5-7 business days to reflect in your account depending on your bank. Could you share your refund confirmation ID? I will check the exact status and escalate if there is any delay on our end."
    },
    {
        "instruction": "How do I cancel my subscription?",
        "response": "To cancel your subscription, go to Account Settings, click on Subscriptions, and select Cancel. Your access will continue until the end of the current billing period. Would you like me to walk you through any specific step?"
    },
    {
        "instruction": "Is there a discount available for students?",
        "response": "Yes! We offer a 20% student discount through our education program. You can apply at our website under the Students section by verifying your enrollment with a valid college email address or student ID."
    },
]

# Now we format each example into the ChatML template
# This is the exact format TinyLlama was trained on, so it understands it
# <|system|> tells the model its role
# <|user|> is the customer's message
# <|assistant|> is what we want the model to learn to say

def format_example(example):
    # This function takes one instruction-response pair and formats it properly
    formatted = f"""<|system|>
You are a helpful, polite, and professional customer support assistant. Always be empathetic and solution-focused.</s>
<|user|>
{example['instruction']}</s>
<|assistant|>
{example['response']}</s>"""
    return {"text": formatted}

# Apply the formatting function to all our examples
# This creates a list of formatted training texts
formatted_data = [format_example(ex) for ex in training_examples]

# Convert our list into a HuggingFace Dataset object
# The trainer needs data in this specific format
dataset = Dataset.from_list(formatted_data)

# Let's check what one example looks like after formatting
print("Sample training example:")
print("-" * 50)
print(dataset[0]["text"])
print("-" * 50)
print(f"\nTotal training examples: {len(dataset)}")

Sample training example:
--------------------------------------------------
<|system|>
You are a helpful, polite, and professional customer support assistant. Always be empathetic and solution-focused.</s>
<|user|>
My order has not arrived yet. It has been 10 days.</s>
<|assistant|>
I apologize for the delay. Could you please share your order ID so I can check the status for you? Standard delivery takes 5-7 business days, but I will investigate this right away.</s>
--------------------------------------------------

Total training examples: 10


## CELL 4 — Load the model and tokenizer

Now I'm loading the pre-trained TinyLlama model and its tokenizer.

I'm using 4-bit quantization when loading the model to reduce its memory requirements and make it practical to run on the Colab GPU.

The tokenizer handles the conversion between human-readable text and the numerical tokens that the model processes.

P.S. I'm just loading the base model and preparing it for the training step.

In [ ]:
# Model to use from the Hugging Face Hub
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure 4-bit quantization to reduce GPU memory usage
# NF4 is used for the 4-bit representation
# Double quantization provides additional memory savings
# Computation is performed in float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Downloading TinyLlama... (this takes 2-4 minutes)")

# Load the model using the quantization configuration
# device_map="auto" allows the available hardware to be used automatically
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Use the EOS token for padding during batch processing
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare the quantized model for parameter-efficient fine-tuning
model = prepare_model_for_kbit_training(model)

print("\nModel loaded successfully!")
print(f"Model is on: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]


Model loaded successfully!
Model is on: cuda:0


## CELL 5 — Setting up LoRA

Here I'm configuring LoRA (Low-Rank Adaptation) for the fine-tuning experiment.

Instead of updating all of TinyLlama's parameters, LoRA keeps the original model weights frozen and adds a small number of trainable parameters to selected layers. This makes the fine-tuning process considerably more memory-efficient.

**LoRA settings**
- r — Controls the rank of the LoRA matrices. A higher value increases the number of trainable parameters
- lora_alpha — Controls the scaling applied to the LoRA updates
- lora_dropout — Applies dropout to the LoRA layers during training, which can help reduce overfitting
- target_modules — Specifies which model layers will receive the LoRA adapters. Here, the attention projection layers are being targeted
- task_type — Specifies the type of task. CAUSAL_LM is used for causal language modeling and text generation

In [ ]:
# Configure the LoRA parameters
lora_config = LoraConfig(
    r=8,                          # LoRA rank
    lora_alpha=16,                # scaling factor
    lora_dropout=0.05,            # dropout to prevent overfitting
    target_modules=[              # which layers to apply LoRA to
        "q_proj",                 # query projection
        "v_proj"                  # value projection
    ],
    bias="none",                  # Keep bias parameters unchanged
    task_type=TaskType.CAUSAL_LM  # Causal language modeling
)

# Applying LoRA to the model

model = get_peft_model(model, lora_config)

# Check the number of trainable parameters
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## CELL 6 — Configure the training

Now I'm setting the parameters that control the fine-tuning process.

The main settings I'm using are:

* num_train_epochs — Number of times the model goes through the training dataset.
* per_device_train_batch_size — Number of training examples processed together on each device.
* learning_rate — Controls the size of the updates made during training.
* max_seq_length — Maximum number of tokens considered for each training example.
* output_dir — Directory where the trained model and related outputs will be saved.

During training, I'll monitor the loss and the training progress to see how the model is responding to the training data.

In [ ]:
# SFTConfig holds all our training settings
# SFT = Supervised Fine-Tuning
training_args = SFTConfig(
    output_dir="./tinyllama-customer-support",  # folder to save the trained model
    num_train_epochs=3,                          # go through all data 3 times
    per_device_train_batch_size=2,               # 2 examples at a time (safe for free GPU)
    gradient_accumulation_steps=4,              # pretend we used batch size 8 (2x4)
    learning_rate=2e-4,                          # step size for learning (0.0002)
    bf16=True,                                   # use 16-bit precision for speed
    logging_steps=5,                             # print loss every 5 steps
    save_steps=50,                               # save model checkpoint every 50 steps
    max_length=512,                         # maximum token length per example
    dataset_text_field="text",                   # which column in our dataset has the text
    report_to="none"                             # don't send logs to wandb (keeps it simple)
)

# SFTTrainer handles the entire training loop by taking: the model, training settings, training data, and tokenizer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

print("Starting training...")
print("Watch the 'loss' value — it should decrease over time.")
print("Lower loss = model is learning better.")
print("-" * 50)

# This single line runs the entire training process
trainer.train()

print("-" * 50)
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Starting training...
Watch the 'loss' value — it should decrease over time.
Lower loss = model is learning better.
--------------------------------------------------


Step,Training Loss
5,1.865245


--------------------------------------------------
Training complete!


## CELL 7 — Save the fine-tuned model

After training, we save the LoRA weights.

Note: we only save the LoRA adapter weights — the tiny matrices we trained.
NOT the full TinyLlama model. This is one of the benefits of LoRA —
the saved file is very small (a few MB) instead of the full 2GB model.

In [ ]:
# Saving only the LoRA adapter weights (not the full base model)
# This is why LoRA is so efficient — the adapter is teeny-tiny
trainer.model.save_pretrained("./tinyllama-customer-support-final")
tokenizer.save_pretrained("./tinyllama-customer-support-final")

print("Model saved to: ./tinyllama-customer-support-final")
print("\nNote: Only the LoRA adapter weights are saved, not the full base model.")
print("This is one of the key advantages of LoRA fine-tuning.")

Model saved to: ./tinyllama-customer-support-final

Note: Only the LoRA adapter weights are saved, not the full base model.
This is one of the key advantages of LoRA fine-tuning.


## CELL 8 — Testing fine-tuned model

Now the exciting part — let's try to talk to the model and see if it learned to be a customer support assistant!

I'll use a few different questions and compare the responses to the behaviour I was aiming for during fine-tuning.

In [ ]:
# Function to generate a response from the fine-tuned model
def get_response(user_message, max_new_tokens=200):
    """
    Takes a user message and returns the model's response.

    Steps happening inside:
    1. Format the message in ChatML template (same as training)
    2. Tokenize it (convert text to numbers)
    3. Pass to model and generate response tokens
    4. Decode the tokens back to text
    5. Return just the assistant's part
    """

    # Format the prompt the same way as in formatted training data
    prompt = f"""<|system|>
You are a helpful, polite, and professional customer support assistant. Always be empathetic and solution-focused.</s>
<|user|>
{user_message}</s>
<|assistant|>
"""

# Tokenize the prompt and move the inputs to the model's device
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate the model's response
# Sampling and temperature control how deterministic the response is
    with torch.no_grad():  # no_grad saves memory during inference (not training here)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode — convert numbers back to text
    # skip_special_tokens=True: removes the <|system|>, <|user|> tags from output
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the assistant's response (everything after the last 'assistant' marker)
    if "assistant" in full_response.lower():
        response_parts = full_response.split("<|assistant|>")
        if len(response_parts) > 1:
            return response_parts[-1].strip()

    return full_response.strip()


# Test it with questions the model has NOT seen during training
test_questions = [
    "I want to track my order.",
    "How do I update my billing information?",
    "My account has been locked."
]

print("Testing fine-tuned model responses:")
print("=" * 60)

for question in test_questions:
    print(f"\nCustomer: {question}")
    print(f"Assistant: {get_response(question)}")
    print("-" * 60)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Testing fine-tuned model responses:

Customer: I want to track my order.
Assistant: Yeschus permissions exercisiltycast jakΆ Раскет « dess angularjs meaning������正 moins queso moins vorcremic Civil permissions jaknéeça « tales weiter angularjs ress «ณえΆ sopchus personn «Лominium program « «furicale coat septcian ress reactjs overrid эксписа мени свядро им Ор Рас эксход « dessils «faces-milędzwürΈшка reactjs logs coatfacescodemsms tales say exactly angularjs nucΈ seule tales such factors feel tales shapes ressnée lettір экс « talesΈณgang reactjs recommatel Werkprec personnied coat reactjs septominΈмель «станказchusgang « dess ressΆмель такой ressณΈчкакет «Marie sug moins galolean «стоястан осу Васильlongrightarrow moins lettณえURI underlying underlying pace reactjs permissions increasing factors directly lengms « « руда marsal angularjs coat angularjs||httpsschemasganghos programmspolatel weiter moins variмельciancianças nabiedominilon amounts
--------------------------------------------

## CELL 9 — Testing with random questions

Testing the fine-tuned model with a few additional prompts to see how it handles questions that were not part of the examples shown during training.

I can modify the prompt and run the cell multiple times to compare the responses.

In [ ]:
# Change this message to anything to check responses
your_question = "I received the wrong item in my order."

print(f"Customer: {your_question}")
print(f"\nAssistant: {get_response(your_question)}")

Customer: I received the wrong item in my order.

Assistant: Itquinied romanätzhoff reactjs nabΆfaces recommarga logs permissions constants underlying permissions, jakfaces «faces: [tradreciätz reactjs «faces logs programmic lett Лючка Павстанçasстояcianal structures worth thou angularjs tales permissionsΈnée quartilon vorwald angularjs nucстанстандии эксход энциклопеди осуспе ressiltypol personncian functions lengominims to feel singing jak suiv seule méfurmelims reactjs���� permissions permissions: Tow coat экспо « resshoffnée réspol angularjs meters lengmail coatwater infrastrкт « tales reactjsÇ underlying constants< leng suiv défemblantly leng эксходпольéoΈшкаΈ reactjs varifaces:output__tool экс так эксф тор Ор Рас священён Василь так seule jak ress Лю мешка Василь «Miltradrecims permissions Civilwerke coatown mij nabfaces__pol permissions meaning tales meaningfullmail angularjs templates reactjs permissions programmeltrad tales of permissions say nabicale logs extensiveness « weit

## Experiment summary

This experiment was to practice fine-tuning a small language model using **LoRA/QLoRA**.

### What I worked through

1. Created a small instruction-response dataset for a customer-support use case.
2. Loaded **TinyLlama 1.1B** with 4-bit quantization to reduce memory usage.
3. Configured **LoRA** using Hugging Face PEFT and applied it to selected attention layers.
4. Fine-tuned the model using **SFTTrainer**.
5. Tested the resulting model with customer-support prompts.
6. Compared its responses to the behaviour I was trying to achieve through the training examples.

One of the main things I wanted to understand was how LoRA allows a relatively small number of parameters to be trained while keeping the original model weights frozen.

### What I learned

The experiment helped me understand the overall fine-tuning workflow, from preparing training examples and configuring the model to training and testing the resulting model.

The small dataset also makes this a useful starting point rather than a production-ready fine-tuning setup. A larger and more diverse dataset would be needed to properly evaluate how well the approach generalizes.
